# The same function, two inner products

**SIAM MDS 2026 minitutorial, exercise B.**

We fit a function $f$ with $f(0)=f(1)=0$ to 30 noisy observations by minimizing

$$J(f)=\tfrac12\sum_{i=1}^{30}\bigl(f(t_i)-y_i\bigr)^2+\tfrac{\gamma}{2}\int_0^1 f'(t)^2\,dt,\qquad\gamma=0.08,$$

over piecewise-linear functions on a mesh with $n$ interior nodes. ROL sees the nodal values $x\in\mathbb R^n$ through a `Vector`. We compare two vectors that hold the same numbers: the Euclidean `NumPyVector` that PyROL provides (the Quick Start's class), with the inner product $a^\top b$, and `H1Vector`, with the $H^1_0$ inner product $a^\top S b$ of the functions they represent.

**Predict first:** we refine the mesh but add no observations. Which should change: the fitted function, the number of iterations, or both?

In [ ]:
%pip install -q --only-binary=:all: "rol-python==2026.11.9.dev1945"

In [ ]:
# ROL prints from C++. Colab needs wurlitzer to show that output in the notebook;
# other Jupyter kernels (e.g. Binder) show it already, and wurlitzer would hide it there.
import sys
if 'google.colab' in sys.modules:
    %pip install -q wurlitzer
    %load_ext wurlitzer

In [ ]:
import sys
from importlib.metadata import version

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import factorized, spsolve
from pyrol import Objective, Problem, Solver, getCout
from pyrol.pyrol import ROL
from pyrol.pyrol.Teuchos import ParameterList
from pyrol.getTypeName import getTypeName

print('Python', sys.version.split()[0], '| rol-python', version('rol-python'))

## 1. The Euclidean `Vector`

PyROL provides the Quick Start's `NumPyVector` as `pyrol.vectors.NumPyVector`. Its inner product is `np.vdot`, and it inherits `dual()`, which returns the vector itself.

In [ ]:
from pyrol.vectors import NumPyVector

## 2. Data and finite elements

Hat functions on $n$ interior nodes give $f_h=\sum_j x_j\varphi_j$. The stiffness matrix $S=h^{-1}\operatorname{tridiag}(-1,2,-1)$ is the $H^1_0$ inner product of hat functions, and $P$ maps nodal values to the values $f_h(t_i)$, so

$$J_h(x)=\tfrac12\|Px-y\|^2+\tfrac{\gamma}{2}x^\top Sx.$$

In [ ]:
# Data: 30 noisy observations of a smooth signal at random times.
rng = np.random.default_rng(20261012)
t_obs = np.sort(rng.uniform(0.03, 0.97, 30))
y_obs = 0.8*np.sin(2*np.pi*t_obs) + 0.27*np.sin(5*np.pi*t_obs) + rng.normal(scale=0.12, size=30)
gamma = 0.08


def stiffness(n):
    """H^1_0 inner product of hat functions on n interior nodes: S = tridiag(-1, 2, -1)/h."""
    h = 1.0/(n + 1)
    return sparse.diags([-np.ones(n-1), 2*np.ones(n), -np.ones(n-1)], (-1, 0, 1), format='csc')/h


def interpolation(n):
    """P maps nodal values x to the values f_h(t_i) at the observation times."""
    h = 1.0/(n + 1)
    j = np.floor(t_obs/h).astype(int)  # t_i lies between nodes j*h and (j+1)*h
    theta = t_obs/h - j
    rows, cols, vals = [], [], []
    for i in range(len(t_obs)):
        for node, weight in ((j[i], 1 - theta[i]), (j[i] + 1, theta[i])):
            if 1 <= node <= n:  # nodes 0 and n+1 carry the boundary zeros
                rows += [i]
                cols += [node - 1]
                vals += [weight]
    return sparse.csr_matrix((vals, (rows, cols)), shape=(len(t_obs), n))

## 3. The $H^1_0$ geometry

`H1Vector.dot` is $a^\top S b$. Its `dual()` applies the Riesz map $x\mapsto Sx$; the dual vector's `dual()` applies $S^{-1}$. The objective's `gradient` writes the derivative $d=P^\top(Px-y)+\gamma Sx$, an element of the dual space, in both geometries; ROL applies `dual()` when it needs a direction in $X$, so the $H^1_0$ gradient is $S^{-1}d$.

In [ ]:
class Riesz:
    """Riesz map of the H^1_0 inner product on nodal values: d -> S^{-1} d and its inverse."""
    def __init__(self, S):
        self.S, self.solve = S, factorized(S)


class H1Vector(NumPyVector):
    """Primal vector: nodal values x of f_h, with (a, b) = a^T S b."""
    def __init__(self, array, riesz):
        super().__init__(array)
        self.riesz, self._dual = riesz, None

    def dot(self, b):
        return float(self.array @ (self.riesz.S @ b.array))

    def apply(self, b):  # the pairing <b, x>: plain coordinates
        return float(self.array @ b.array)

    def clone(self):
        return H1Vector(np.zeros_like(self.array), self.riesz)

    def dual(self):  # inverse Riesz map x -> S x, into one stored object
        if self._dual is None:
            self._dual = H1DualVector(0*self.array, self.riesz)
        self._dual.array[:] = self.riesz.S @ self.array
        return self._dual


class H1DualVector(NumPyVector):
    """Dual vector: derivative coordinates d, with (d, e) = d^T S^{-1} e."""
    def __init__(self, array, riesz):
        super().__init__(array)
        self.riesz, self._dual = riesz, None

    def dot(self, b):
        return float(self.array @ self.riesz.solve(b.array))

    def apply(self, b):
        return float(self.array @ b.array)

    def clone(self):
        return H1DualVector(np.zeros_like(self.array), self.riesz)

    def dual(self):  # Riesz map d -> S^{-1} d
        if self._dual is None:
            self._dual = H1Vector(0*self.array, self.riesz)
        self._dual.array[:] = self.riesz.solve(self.array)
        return self._dual

In [ ]:
class SmoothingObjective(Objective):
    """J_h(x) = 1/2 ||P x - y||^2 + gamma/2 x^T S x on nodal values x."""
    def __init__(self, n):
        super().__init__()
        self.P, self.S = interpolation(n), stiffness(n)

    def value(self, x, tol):
        r = self.P @ x.array - y_obs
        return 0.5*(r @ r) + 0.5*gamma*(x.array @ (self.S @ x.array))

    def gradient(self, g, x, tol):  # the derivative d, in X*
        r = self.P @ x.array - y_obs
        g.array[:] = self.P.T @ r + gamma*(self.S @ x.array)

    def hessVec(self, hv, v, x, tol):
        hv.array[:] = self.P.T @ (self.P @ v.array) + gamma*(self.S @ v.array)

## 4. The same stopping rule in both geometries

The two vectors report different gradient norms for the same iterate, so a gradient tolerance would stop them at different points. Instead, the runs stop on the relative objective gap $(J(x_k)-J_*)/(J(x_0)-J_*)\le10^{-7}$, with $J_*$ from a direct sparse solve.

**ROL supports user-defined stopping conditions.** Subclass ROL's `StatusTest` (`getTypeName('StatusTest')` returns PyROL's class for it) and implement `check(state)`: it receives the algorithm's state (`state.iter`, `state.value`, `state.gnorm`, ...) at every iteration and returns `True` to continue or `False` to stop. `RelativeGap` stops at the target gap and sets the exit status to *user defined*. `solver.solve(stream, stop, True)` combines it with ROL's own tests from the `ParameterList`, so the iteration limit still applies; with `False`, only `stop` decides.

In [ ]:
class RelativeGap(getTypeName('StatusTest')):
    """Stop when (J(x_k) - J*)/(J(x_0) - J*) <= target, the same test in either geometry."""
    def __init__(self, j_star, target):
        super().__init__()
        self.j_star, self.target, self.j0 = j_star, target, None

    def check(self, state):
        if self.j0 is None:
            self.j0 = state.value
        if state.value - self.j_star > self.target*(self.j0 - self.j_star):
            return True
        state.statusFlag = ROL.EExitStatus.EXITSTATUS_USERDEFINED
        return False


def parameters(method, limit, output=1):
    params = ParameterList()
    params['General'] = ParameterList()
    params['General']['Output Level'] = output  # 0: no iteration table
    params['Step'] = ParameterList()
    if method == 'steepest descent':
        params['Step']['Type'] = 'Line Search'
        params['Step']['Line Search'] = ParameterList()
        params['Step']['Line Search']['Descent Method'] = ParameterList()
        params['Step']['Line Search']['Descent Method']['Type'] = 'Steepest Descent'
        params['Step']['Line Search']['Line-Search Method'] = ParameterList()
        params['Step']['Line Search']['Line-Search Method']['Type'] = 'Cubic Interpolation'
    else:
        params['Step']['Type'] = 'Trust Region'
        params['Step']['Trust Region'] = ParameterList()
        params['Step']['Trust Region']['Subproblem Solver'] = 'Truncated CG'
    params['Status Test'] = ParameterList()
    params['Status Test']['Gradient Tolerance'] = 1e-14  # RelativeGap stops the run
    params['Status Test']['Step Tolerance'] = 1e-30
    params['Status Test']['Iteration Limit'] = limit
    return params


def solve(n, geometry, method='steepest descent', target=1e-7, limit=1500, output=1):
    objective = SmoothingObjective(n)
    if geometry == 'H1':
        x = H1Vector(np.zeros(n), Riesz(objective.S))
    else:
        x = NumPyVector(np.zeros(n))
    x_star = spsolve((objective.P.T @ objective.P + gamma*objective.S).tocsc(), objective.P.T @ y_obs)
    stop = RelativeGap(objective.value(NumPyVector(x_star), 0.0), target)
    solver = Solver(Problem(objective, x), parameters(method, limit, output))
    stream = getCout()
    solver.solve(stream, stop, True)
    return x.array.copy(), solver.getAlgorithmState()  # state.iter, .statusFlag

## 5. One solve with ROL's iteration table

In [ ]:
x, state = solve(128, 'H1')
print('iterations:', state.iter, '| status:', state.statusFlag)

## 6. The experiment

Run both geometries on three meshes. `output=0` suppresses the tables; the Euclidean run on 512 nodes stops at the iteration limit of 1500.

In [ ]:
fits = {}
for n in (32, 128, 512):
    for geometry in ('H1', 'Euclidean'):
        x, state = solve(n, geometry, output=0)
        fits[n, geometry] = x
        print(f'n = {n:4d}  {geometry:9s}  iterations = {state.iter:4d}  ({state.statusFlag})')

## 7. Did the fitted function change?

Compare the $H^1_0$ fits on the three meshes (each reached the same relative accuracy).

In [ ]:
import matplotlib.pyplot as plt

t = np.linspace(0, 1, 1001)
def as_function(x):
    n = len(x)
    return np.interp(t, np.r_[0, np.arange(1, n + 1)/(n + 1), 1], np.r_[0, x, 0])

for n in (32, 128, 512):
    plt.plot(t, as_function(fits[n, 'H1']), label=f'n = {n}')
plt.scatter(t_obs, y_obs, s=10, c='k', label='data')
plt.legend(); plt.xlabel('t'); plt.show()
print('max |f_512 - f_128| =', np.abs(as_function(fits[512, 'H1']) - as_function(fits[128, 'H1'])).max())

## Questions

1. Both geometries minimize the same $J_h$. Why does the Euclidean iteration count grow with $n$?
2. For the same iterate, the `gnorm` column differs between the geometries. Which norm is it?
3. *Optional:* run `solve(512, 'H1', 'trust region')` and `solve(512, 'Euclidean', 'trust region', limit=100)` and compare the `iterCG` column.